In [1]:
import numpy as np
import cv2
import cMonocularMeasure
from ccalibration import CCameraCalibration

In [2]:
#多张图片标定得到相机的内参与畸变向量
images_path = r'C:\Users\hp\Desktop\mm'
mycc = CCameraCalibration()
mycc.monocular_set_calibration_images(images_path, 'monocular.npy', False, 11, 720, 1280, [7,7], 5.0, mode='circle')
mycc.monocular_calibration('monocular.npy', True)
mycc.monocular_print()

You do not have a calibration file

 Internal matrix IS 
 [[5.09780710e+03 0.00000000e+00 6.98983123e+02]
 [0.00000000e+00 5.09433703e+03 3.47929859e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]] 

Distortion vector is 
 [[-1.78303100e+00  1.10415663e+02 -2.45426438e-03  2.22971601e-03
  -2.26854161e+03]] 



In [3]:
mycmm = cMonocularMeasure.MonocularMeasure()
#对称圆标定板上圆的行数、列数
monocular_circleSize = [7,7]
monocular_circleCellLen = 5.0 #mm，圆心距

#棋盘格
# monocular_circleSize = [11, 8]
# monocular_circleCellLen = 3.0 #格子大小
mode = 'square'
mode = 'circle'

In [12]:
#测距用的单张标定图像，该图不需要畸变矫正
imgpath = 'C:\\Users\\hp\\Desktop\\1.jpg' #yl
# 得到该标定图像中标定板上的圆心实际坐标与对应像素坐标
ret_l, world_points, pixel_points = mycmm.findCornersAndWorldPoints(singleImgPath=imgpath, 
                                                chessSize=monocular_circleSize, 
                                                chessCellLen=monocular_circleCellLen, mode=mode)
# h, mask = cv2.findHomography( pixel_points, world_points[:,:-1],method=cv2.RANSAC)

A = mycc._monocularParameters["cameraMatrix"]
distCoeff = mycc._monocularParameters["distCoeffs"]
h_low = -3.0
h = mycmm.findHByPnPAndZDiff(A, distCoeff, world_points, pixel_points, zLow=h_low)

AttributeError: 'NoneType' object has no attribute 'reshape'

In [8]:
print(h)

[[ 6.53343260e-02 -5.99739502e-02 -1.52708599e+01]
 [ 5.99399694e-02  6.54105148e-02 -5.98324262e+01]
 [-7.07879374e-06 -9.62438522e-07  1.00000000e+00]]


In [11]:
#zs
p0 = np.array([353, 268, 1], dtype=np.float32).reshape(3,1) #从畸变矫正后的图里面量取
p1 = np.array([680, 15, 1], dtype=np.float32).reshape(3,1)

################################################

#求世界坐标平面上的齐次坐标，并做归一化(一般可以不用)
pt1 = h @ p0
pt1 = pt1/pt1[-1]

pt2 = h @ p1
pt2 = pt2/pt2[-1]

length = np.math.sqrt(np.sum((pt1-pt2)**2)) #求解两点长度

pix_len = np.math.sqrt(np.sum((p0-p1)**2))
print('H Matrix : ','\n', h, '\n')
print('source point: ', p0.reshape(-1), p1.reshape(-1))
print('dst point: ', pt1.reshape(-1), pt2.reshape(-1))
print('\nlen: ', length)
print('pixel len:', pix_len)

H Matrix :  
 [[ 6.61973609e-02 -6.07662807e-02 -1.54587188e+01]
 [ 6.07310955e-02  6.62745536e-02 -6.05180294e+01]
 [-7.07879374e-06 -9.62438522e-07  1.00000000e+00]] 

source point:  [353. 268.   1.] [680.  15.   1.]
dst point:  [ -8.3995691  -21.37730416   1.        ] [ 28.78295701 -18.31519221   1.        ]

len:  37.30840089496921
pixel len: 413.4464898871437
